# 🛠️ Workshop 4 — Building Cooperative LLM Agent Workflows
## Anti-pattern Detection · Code Smells · Technical Debt Resolution

**LLMA4SE 2026** — 2nd International Summer School on LLM-based Agents for Software Engineering
Day 3 · 15:00 – 18:00 · Instructors: **Karthik Shivashankar** (SINTEF Digital / UiO) & **Adela Nedisan Videsjorden** (UiO)

---

### What you will build today

| Part | You build | Time |
|---|---|---|
| **0** | Setup: the toolbox + your OpenAI-powered brain | 10 min |
| **1** | 🕵️ **The Code Auditor** — one agent, four deterministic tools (radon · pylint · PyExamine · MLScent) | 50 min |
| **2** | 🤝 **The Refactoring Team** — Auditor → Refactorer ⇄ QA, a blackboard, verification gates, a failure lab | 60 min |
| **3** | 💳 **The Technical-Debt Pipeline** — classify, triage, resolve, report | 40 min |
| **4** | 🚀 **Production frameworks** — the same team in LangGraph, an autonomous Deep Agent, and a packaged CLI | bonus |

### Runtime

**No GPU needed.** `Runtime → Change runtime type → CPU` is fine — the brain lives behind the **OpenAI API**.
You need one thing: an **OpenAI API key**.

> 💡 **The one idea to take home:** *deterministic tools measure, the LLM interprets, and a gate decides.*
> Everything else in this notebook is plumbing around that sentence.

---
# Part 0 · Setup

## 0.1 · Install the toolbox (~2 min)

| Package | What it is | Why an agent needs it |
|---|---|---|
| `radon` | complexity & maintainability metrics | fast, deterministic **eyes** |
| `pylint` | classic rule-based linter | a second pair of eyes with different blind spots |
| `code-quality-analyzer` | **PyExamine** (MSR 2025) — 49 metrics, 3 levels | research-grade smell detection |
| `ml-code-smell-detector` | **MLScent** (CAIN 2025) — 76 ML anti-pattern detectors | smells that only exist in ML code |
| `pytest` | test runner | the **QA gate** — the only reason we can trust an agent's patch |
| `pandas` | dataframes | scoring the debt classifier in Part 3 |
| `openai` | OpenAI SDK | the agent's **brain** |
| `langgraph`, `deepagents` | agent frameworks | Part 4 — the production rebuild |

In [ ]:
%pip install -q code-quality-analyzer ml-code-smell-detector radon pylint pytest pandas langgraph deepagents openai
print("✅ toolbox installed")

## 0.2 · Your OpenAI API key 🔑

Three ways to provide it — the loader below tries them **in order** and stops at the first that works:

1. **`.env` file** — upload a file called `.env` (Colab left sidebar → 📁 → upload) containing one line:
   `OPENAI_API_KEY=sk-...`
2. **Colab Secrets** — 🔑 icon in the left sidebar → add `OPENAI_API_KEY` → toggle *Notebook access*. *(Recommended: it never lands in the notebook file.)*
3. **Typed prompt** — a hidden `getpass` box, as a last resort.

> ⚠️ **Never paste a key into a code cell.** Notebooks get shared, committed and screenshotted. This is not paranoia — it is the #1 way keys leak.

In [ ]:
import os, getpass, pathlib

def load_openai_key() -> str:
    """.env  →  Colab secret  →  typed prompt. First hit wins."""
    if os.environ.get("OPENAI_API_KEY"):
        print("🔑 key found in the environment")
        return os.environ["OPENAI_API_KEY"]

    # 1) a .env file (a few lines of stdlib beat a dependency)
    for candidate in (pathlib.Path("/content/.env"), pathlib.Path(".env")):
        if not candidate.exists():
            continue
        for line in candidate.read_text().splitlines():
            line = line.strip()
            if line.startswith("#") or "=" not in line:
                continue
            name, value = line.split("=", 1)
            os.environ[name.strip()] = value.strip().strip("'\"")   # picks up OPENAI_MODEL too
        if os.environ.get("OPENAI_API_KEY"):
            print(f"🔑 key loaded from {candidate}")
            return os.environ["OPENAI_API_KEY"]

    # 2) Colab Secrets
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            print("🔑 key loaded from Colab Secrets")
            return key
    except Exception:
        pass

    # 3) ask
    key = getpass.getpass("Paste your OpenAI API key (input hidden): ").strip()
    os.environ["OPENAI_API_KEY"] = key
    print("🔑 key set for this session")
    return key

_ = load_openai_key()
print("key ends with …" + _[-4:] if _ else "❌ no key!")

## 0.3 · The brain: one `llm()` function

Every agent in this notebook — all six of them — calls **this one function**. That is deliberate: swapping the
model, the provider or the temperature is a one-line change, and the rest of the workshop cannot tell
the difference. It also absorbs the one API wrinkle you will hit in practice: **reasoning models**
(`gpt-5.x`, `o*`) reject `temperature` and rename `max_tokens`.

We also keep a **cost meter**, because "how much did my agent just spend?" is a production question, not an afterthought.

In [ ]:
import os
from openai import OpenAI

client = OpenAI()                     # reads OPENAI_API_KEY from the environment

# Your .env may pin a model with OPENAI_MODEL=...; otherwise this cheap, capable default is used.
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
#   "gpt-4.1-mini"  · good code model, cheap, fast          ← recommended for a live workshop
#   "gpt-4o-mini"   · cheapest, weakest at long rewrites
#   "gpt-4.1"       · strongest classic model here
#   "gpt-5.x" / "o*" · reasoning models: slower, no temperature knob, but the best refactorers

# Reasoning models rename max_tokens and reject temperature. Detect once, adapt everywhere.
IS_REASONING = MODEL_NAME.startswith(("gpt-5", "o1", "o3", "o4"))
print(f"🧠 brain: {MODEL_NAME}" + ("  (reasoning model)" if IS_REASONING else ""))

USAGE = {"calls": 0, "in": 0, "out": 0}          # the cost meter


def llm(user_prompt: str,
        system_prompt: str = "You are a helpful assistant.",
        max_new_tokens: int = 1024,
        temperature: float = 0.2,
        json_mode: bool = False) -> str:
    """One call to the OpenAI API. EVERY agent in this workshop goes through here."""
    kwargs = dict(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
    )
    if IS_REASONING:
        # reasoning tokens are billed out of the same budget, so give it room to think
        kwargs["max_completion_tokens"] = max(max_new_tokens * 4, 4000)
    else:
        kwargs["max_tokens"] = max_new_tokens
        kwargs["temperature"] = temperature
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    try:
        resp = client.chat.completions.create(**kwargs)
    except Exception as e:                       # last-ditch adaptation for unknown models
        print(f"   ↻ retrying without model-specific params ({type(e).__name__})")
        kwargs.pop("temperature", None)
        if "max_tokens" in kwargs:
            kwargs["max_completion_tokens"] = max(kwargs.pop("max_tokens") * 4, 4000)
        resp = client.chat.completions.create(**kwargs)

    USAGE["calls"] += 1
    USAGE["in"] += resp.usage.prompt_tokens
    USAGE["out"] += resp.usage.completion_tokens
    return (resp.choices[0].message.content or "").strip()


def cost_report(price_in: float = 0.40, price_out: float = 1.60):
    """Rough $ estimate. Defaults are gpt-4.1-mini list prices, $ per 1M tokens."""
    usd = USAGE["in"] / 1e6 * price_in + USAGE["out"] / 1e6 * price_out
    print(f"📊 {USAGE['calls']} calls · {USAGE['in']:,} in / {USAGE['out']:,} out tokens · ≈ ${usd:.4f}")


# Smoke test — is the brain alive?
print(llm("In one sentence: what is a code smell?"))
cost_report()

> 🧪 **Try it (30 s):** re-run the cell with `temperature=1.5`. Then `0.0`. Which setting do you want for a
> *refactoring* agent, and why? (Hold that thought — Part 2 depends on the answer.)

## 0.4 · A clean workspace

Everything the agents create lives in `/content/workshop`, so the static analysers never wander into Colab's
`sample_data/` folder.

In [ ]:
import os, pathlib
WORKDIR = (pathlib.Path("/content/workshop") if pathlib.Path("/content").exists()
           else pathlib.Path("./workshop")).resolve()
WORKDIR.mkdir(exist_ok=True)
os.chdir(WORKDIR)
(WORKDIR / "ml_project").mkdir(exist_ok=True)
print("📁 working in", os.getcwd())

---
# Part 1 · 🕵️ The Code Auditor Agent

**⏱ ~50 min**

One agent. Four tools. One JSON contract. By the end of this part you will have an agent that finds real
smells in real code and hands them to the next agent in a machine-readable form.

## 1.1 · Meet the patient 🤒

`inventory.py` is a small order-processing module that **works perfectly** — every test passes — yet it is
riddled with classic smells:

mutable default argument · dead code · long parameter list · long method · deep nesting · magic numbers · duplicated logic

> 🎯 **Look before you scroll:** give yourself 60 seconds to spot three of them by eye. That is the baseline
> your agent has to beat.

In [ ]:
%%writefile inventory.py
"""inventory.py -- Order processing for a small e-commerce shop.

This module works correctly (all tests pass!) but it is deliberately
full of code smells. Your agents will find and fix them.
"""

def helper_unused(x):          # SMELL: dead code -- never called anywhere
    return x * 2


class InventoryManager:
    """Manages stock and processes customer orders."""

    def __init__(self, items=[]):              # SMELL: mutable default argument
        self.items = {}
        for name, price, qty in items:
            self.items[name] = {"price": price, "qty": qty}
        self.log = []

    def add_item(self, name, price, qty, category, supplier, discount, taxable):
        # SMELL: long parameter list (7 params, most unused)
        self.items[name] = {"price": price, "qty": qty}
        return True

    def process_order(self, order):
        # SMELL: long method, deep nesting, magic numbers, duplication
        total = 0.0
        status = "ok"
        for name, qty in order:
            if name in self.items:
                if self.items[name]["qty"] >= qty:
                    if qty > 0:
                        price = self.items[name]["price"]
                        subtotal = price * qty
                        if subtotal > 100:                      # magic number
                            subtotal = subtotal - subtotal * 0.05   # magic number
                        if qty > 10:                            # magic number
                            subtotal = subtotal - subtotal * 0.02   # magic number
                        total = total + subtotal
                        self.items[name]["qty"] = self.items[name]["qty"] - qty
                        self.log.append("sold " + name)
                    else:
                        status = "invalid_qty"
                else:
                    status = "insufficient_stock"
            else:
                status = "unknown_item"
        total = total + total * 0.25            # magic number (VAT)
        return {"total": round(total, 2), "status": status}

    def refund_order(self, order):
        # SMELL: duplicated logic (mirror of process_order maths)
        total = 0.0
        for name, qty in order:
            if name in self.items:
                price = self.items[name]["price"]
                subtotal = price * qty
                if subtotal > 100:                              # magic number again
                    subtotal = subtotal - subtotal * 0.05
                if qty > 10:
                    subtotal = subtotal - subtotal * 0.02
                total = total + subtotal
                self.items[name]["qty"] = self.items[name]["qty"] + qty
        total = total + total * 0.25
        return {"total": round(total, 2), "status": "refunded"}

    def get_stock(self, name):
        if name in self.items:
            return self.items[name]["qty"]
        return 0

### The safety net 🥅

Before we let *any* AI touch this code, we pin down its behaviour with tests. These tests are the **contract**:
a refactoring is valid only if they stay green.

> 💬 **Discuss (30 s with your neighbour):** why must the tests exist *before* the refactoring agent runs,
> and not be written by the agent afterwards?

In [ ]:
%%writefile test_inventory.py
"""test_inventory.py -- Behaviour-preserving safety net.

These tests define the PUBLIC CONTRACT of the module. Any refactoring
your agents perform MUST keep every one of these green.
"""
import pytest
from inventory import InventoryManager


@pytest.fixture
def mgr():
    return InventoryManager([("widget", 10.0, 100), ("gizmo", 25.0, 5)])


def test_simple_order(mgr):
    result = mgr.process_order([("widget", 2)])
    assert result["status"] == "ok"
    assert result["total"] == 25.0          # 20 + 25% VAT


def test_bulk_discount_applied(mgr):
    # 20 widgets = 200 -> -5% (>100) -> -2% (>10 units) -> +25% VAT
    result = mgr.process_order([("widget", 20)])
    assert result["total"] == 232.75


def test_stock_is_decremented(mgr):
    mgr.process_order([("widget", 2)])
    assert mgr.get_stock("widget") == 98


def test_insufficient_stock(mgr):
    result = mgr.process_order([("gizmo", 99)])
    assert result["status"] == "insufficient_stock"


def test_unknown_item(mgr):
    result = mgr.process_order([("nonexistent", 1)])
    assert result["status"] == "unknown_item"


def test_refund_restores_stock(mgr):
    mgr.process_order([("widget", 2)])
    mgr.refund_order([("widget", 2)])
    assert mgr.get_stock("widget") == 100


def test_no_shared_state_between_instances():
    a = InventoryManager()
    b = InventoryManager()
    a.items["x"] = {"price": 1, "qty": 1}
    assert "x" not in b.items or a.items is not b.items

In [ ]:
# The smelly code WORKS — that's the whole point:
!python -m pytest test_inventory.py -q

## 1.2 · Deterministic tools first — the agent's "eyes" 👀

A core design rule of agentic software engineering:

> ⚖️ **Never make an LLM guess what a deterministic tool can measure.**
> Static analysers are fast, cheap, and never hallucinate. The LLM's job is *interpretation and action*, not *measurement*.

### Tool 1 — `radon`: complexity metrics

In [ ]:
import subprocess, json, pathlib

def run_radon(path: str) -> str:
    """Cyclomatic complexity (CC) per function + Maintainability Index (MI)."""
    cc = subprocess.run(["radon", "cc", "-s", path], capture_output=True, text=True).stdout
    mi = subprocess.run(["radon", "mi", "-s", path], capture_output=True, text=True).stdout
    return f"CYCLOMATIC COMPLEXITY (A=best, F=worst):\n{cc}\nMAINTAINABILITY INDEX (100=best):\n{mi}"

print(run_radon("inventory.py"))

📊 **Read the output:** `process_order` should stand out. CC counts independent paths through a function —
every `if` adds one. High CC = hard to test, hard to change.

### Tool 2 — `pylint`: rule-based linting

In [ ]:
def run_pylint(path: str, max_findings: int = 15) -> str:
    """Classic linter findings as compact text."""
    raw = subprocess.run(
        ["pylint", path, "--output-format=json", "--disable=C0114,C0115,C0116"],
        capture_output=True, text=True
    ).stdout
    try:
        issues = json.loads(raw)[:max_findings]
    except json.JSONDecodeError:
        return raw[:1500]
    return "\n".join(f"L{i['line']}: [{i['symbol']}] {i['message']}" for i in issues) or "no findings"

print(run_pylint("inventory.py"))

👀 Notice pylint catches the **mutable default argument** (`dangerous-default-value`) — a bug-in-waiting that
radon's metrics are blind to. Different tools see different smells. That is why real auditors combine them.

### Tool 3 — **PyExamine** 🔬 (research tool · MSR 2025)

PyExamine analyses code at **three levels** — code, structural, architectural — across 49 metrics.
`pip install code-quality-analyzer`

In [ ]:
def run_pyexamine(directory: str = ".") -> str:
    """PyExamine: multi-level smell detection (Shivashankar & Martini, MSR 2025).
    Findings are written to <output>.txt, so we run the CLI then read the report."""
    subprocess.run(
        ["analyze_code_quality", directory, "--type", "code",
         "--output", "pyexamine_report",
         "--ignore", "sample_data", ".config", "__pycache__"],
        capture_output=True, text=True, timeout=600,
    )
    report = pathlib.Path("pyexamine_report.txt")
    return report.read_text()[:3000] if report.exists() else "PyExamine produced no report"

print(run_pyexamine("."))

> 🧪 **Try it (2 min):** re-run with `--type structural`. Which *new* smells appear that the code-level pass missed?

### Tool 4 — **MLScent** 🤖 (smells that exist only in ML code · CAIN 2025)

Classic tools are blind to a whole category of rot that lives *only* in machine-learning code: NaN comparisons,
missing `zero_grad()`, unseeded randomness, data leakage. MLScent ships **76 detectors** for
PyTorch / TensorFlow / sklearn / pandas / numpy / HuggingFace.
`pip install ml-code-smell-detector`

In [ ]:
%%writefile ml_project/train_model.py
"""train_model.py -- Churn-prediction training script.

It trains fine... but is it reproducible? Is it healthy ML code?
Your ML Auditor agent (powered by MLScent) will tell you.
"""
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def load_data(path):
    df = pd.read_csv(path)
    for i in range(len(df)):                      # pandas: unnecessary iteration
        if df["age"][i] == np.nan:                # numpy: NaN equality (always False!)
            df["age"][i] = 0                      # pandas: chain indexing
    return df


def train():
    df = load_data("churn.csv")
    X = df.drop("label", axis=1).values
    y = df["label"].values
    # sklearn: no feature scaling, no pipeline, no random_state
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

    model = nn.Sequential(nn.Linear(X.shape[1], 64), nn.ReLU(), nn.Linear(64, 2))
    opt = torch.optim.Adam(model.parameters(), lr=0.003)   # hardcoded hyperparams
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(100):                      # no early stopping, no checkpoints
        out = model(torch.tensor(X_train, dtype=torch.float32))
        loss = loss_fn(out, torch.tensor(y_train))
        loss.backward()                           # pytorch: missing opt.zero_grad()
        opt.step()

    preds = model(torch.tensor(X_test, dtype=torch.float32)).argmax(1).numpy()
    print("accuracy:", accuracy_score(y_test, preds))   # over-reliance on accuracy
    # no torch.manual_seed / np.random.seed anywhere -> unreproducible


if __name__ == "__main__":
    train()

> 😱 Before running the detector: **spot 3 smells yourself** (60 seconds, no scrolling back).

Now wrap MLScent as a tool, exactly like the others. Note it never *runs* the script — it is pure AST analysis,
so it is safe on code you have never seen.

In [ ]:
def run_mlscent(project_dir: str = "ml_project") -> str:
    """MLScent: 76 ML-specific anti-pattern detectors (Shivashankar, CAIN 2025).
    Findings land in output/analysis_report.txt, so we run the CLI then read it."""
    subprocess.run(["ml_smell_detector", "analyze", project_dir],
                   capture_output=True, text=True, timeout=600)
    report = pathlib.Path("output/analysis_report.txt")
    return report.read_text()[:3500] if report.exists() else "MLScent produced no report"

print(run_mlscent("ml_project"))

📊 **Read the output:** MLScent flags the `== np.nan` bug (a *correctness* problem — that branch **never**
executes) and the missing `zero_grad()` (gradients accumulate across epochs → wrong training). Note each finding
ships a **"How to fix"** — that is structured, actionable input for an LLM, not just a complaint.

## 1.3 · Anatomy of an agent

Here is our minimal, framework-free agent. Production frameworks (LangGraph, AutoGen, CrewAI) add scheduling,
persistence and tracing — but **this is the essential skeleton they all share**:

| Ingredient | In our class | In production frameworks |
|---|---|---|
| **Role** | `system_prompt` | agent persona / instructions |
| **Brain** | `llm()` | model client |
| **Tools** | `{name: callable}` | tool registry / function calling |
| **Contract** | "reply ONLY with JSON" | structured output / schemas |

In [ ]:
class Agent:
    """Minimal agent: a role + a brain + tools + an output contract."""

    def __init__(self, name: str, system_prompt: str, tools: dict | None = None):
        self.name = name
        self.system_prompt = system_prompt
        self.tools = tools or {}          # {"tool_name": callable}

    def use_tools(self, *args) -> str:
        """Run every tool and concatenate the evidence."""
        report = []
        for tool_name, fn in self.tools.items():
            print(f"  🔧 {self.name} is running tool: {tool_name}")
            try:
                report.append(f"=== {tool_name} ===\n{fn(*args)}")
            except Exception as e:
                report.append(f"=== {tool_name} FAILED: {e} ===")
        return "\n\n".join(report)

    def think(self, prompt: str, **kw) -> str:
        return llm(prompt, system_prompt=self.system_prompt, **kw)

print("✅ Agent class ready")

## 1.4 · The Code Auditor

Its workflow: **(1)** run all tools → **(2)** hand the raw evidence + source to the LLM → **(3)** demand structured JSON.

Two prompt-engineering moves make the result reliable:
1. **Evidence-grounding** — the LLM *summarises tool output* instead of inventing findings from memory.
2. **A hard output contract** — JSON mode, plus a defensive parser, so the next agent can consume the result mechanically.

In [ ]:
import re

AUDITOR_PROMPT = """You are a meticulous senior code reviewer.
You receive: (a) a Python source file, (b) evidence from static-analysis tools.
Identify the most important code smells / anti-patterns.

Reply with a JSON object of the form {"findings": [...]}, where each element is:
{"smell": "<short name>", "location": "<function/line>",
 "severity": "high|medium|low", "why": "<one sentence>",
 "fix": "<one-sentence refactoring suggestion>"}
List at most 6 findings, most severe first."""


def extract_json(text: str):
    """LLMs love to wrap JSON in chatter — dig the array/object out."""
    m = re.search(r"\[.*\]|\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError(f"No JSON found in: {text[:200]}")
    data = json.loads(m.group(0))
    return data.get("findings", data) if isinstance(data, dict) else data


auditor = Agent(
    name="Code Auditor",
    system_prompt=AUDITOR_PROMPT,
    tools={"radon": run_radon, "pylint": run_pylint},
)


def audit(path: str) -> list[dict]:
    evidence = auditor.use_tools(path)
    source = open(path).read()
    raw = auditor.think(
        f"SOURCE FILE ({path}):\n```python\n{source}\n```\n\n"
        f"TOOL EVIDENCE:\n{evidence}\n\nProduce the JSON findings now.",
        max_new_tokens=900, temperature=0.1, json_mode=True,
    )
    return extract_json(raw)


findings = audit("inventory.py")
findings

In [ ]:
# Pretty-print the audit like a review dashboard
SEV = {"high": "🔴", "medium": "🟠", "low": "🟡"}
print(f"{'':2} {'SMELL':28} {'WHERE':22} WHY")
print("-" * 95)
for f in findings:
    print(f"{SEV.get(f.get('severity','low'),'⚪')} {f.get('smell','?'):28.28} "
          f"{str(f.get('location','?')):22.22} {f.get('why','')[:40]}")
    print(f"{'':2} {'↳ fix:':28} {f.get('fix','')[:60]}")
cost_report()

🎉 **You built your first agent.** Deterministic tools did the measuring, the LLM did the interpreting, and a
JSON contract made the result machine-readable — ready to hand to the *next* agent in Part 2.

## 1.5 · Same skeleton, different eyes: the **ML Auditor** 🤖

Change the tools and the prompt; keep the architecture. That reuse *is* the point of the `Agent` abstraction.

In [ ]:
ML_AUDITOR_PROMPT = """You are a senior ML engineer reviewing training code.
You receive: (a) an ML training script, (b) evidence from MLScent, a static
analyser with 76 ML-specific detectors.
Prioritise: (1) silent correctness bugs, (2) reproducibility,
(3) training hygiene (early stopping, checkpoints, eval mode), (4) style.

Reply with a JSON object {"findings": [...]}, each element:
{"smell": "<short name>", "impact": "correctness|reproducibility|hygiene|style",
 "severity": "high|medium|low", "why": "<one sentence>",
 "fix": "<one-sentence fix>"}
List at most 6 findings, most severe first."""

ml_auditor = Agent(
    name="ML Auditor",
    system_prompt=ML_AUDITOR_PROMPT,
    tools={"mlscent": run_mlscent},
)


def ml_audit(project_dir: str, main_file: str) -> list[dict]:
    evidence = ml_auditor.use_tools(project_dir)
    source = open(main_file).read()
    raw = ml_auditor.think(
        f"TRAINING SCRIPT ({main_file}):\n```python\n{source}\n```\n\n"
        f"MLSCENT EVIDENCE:\n{evidence}\n\nProduce the JSON findings now.",
        max_new_tokens=900, temperature=0.1, json_mode=True,
    )
    return extract_json(raw)


ml_findings = ml_audit("ml_project", "ml_project/train_model.py")

IMPACT = {"correctness": "💥", "reproducibility": "🎲", "hygiene": "🧼", "style": "✏️"}
for f in ml_findings:
    print(f"{IMPACT.get(f.get('impact','style'),'•')} [{f.get('severity','?'):6}] "
          f"{f.get('smell','?')}: {f.get('why','')}")
    print(f"      ↳ {f.get('fix','')}")

> 💬 **Discuss (1 min):** the `== np.nan` bug means the data-cleaning branch *never runs* — yet the script trains
> and prints an accuracy. Would a code review catch it? Would your CI? What does that say about where ML
> technical debt hides?

## 1.6 · ✍️ Exercise 1 (10 min) — give the auditor a new sense

The auditor is blind to **magic numbers** as such: pylint does not flag them and radon only counts branches.
Write the missing tool, register it, and re-run the audit.

In [ ]:
import ast

def find_magic_numbers(path: str) -> str:
    """Report numeric literals that are not 0, 1, or -1 (with line numbers)."""
    tree = ast.parse(open(path).read())
    hits = []
    for node in ast.walk(tree):
        # TODO 1: check `isinstance(node, ast.Constant)` and that node.value
        #         is an int/float not in {0, 1, -1}
        # TODO 2: append f"L{node.lineno}: magic number {node.value}" to hits
        pass
    return "\n".join(hits) if hits else "no magic numbers found"

# TODO 3: register the tool and re-audit
# auditor.tools["magic_numbers"] = find_magic_numbers
# findings = audit("inventory.py")

print(find_magic_numbers("inventory.py"))

<details><summary>💡 Click for the solution</summary>

```python
def find_magic_numbers(path: str) -> str:
    tree = ast.parse(open(path).read())
    hits = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
                and node.value not in (0, 1, -1) and not isinstance(node.value, bool):
            hits.append(f"L{node.lineno}: magic number {node.value}")
    return "\n".join(hits) if hits else "no magic numbers found"

auditor.tools["magic_numbers"] = find_magic_numbers
findings = audit("inventory.py")
```
</details>

> 🧠 **The lesson behind the exercise:** you just extended an agent's *perception* without touching its brain,
> its prompt or its contract. In agentic systems, capability usually grows through **tools**, not through
> bigger models.

In [ ]:
# Save the findings — Part 2 picks them up from here
with open("findings.json", "w") as fh:
    json.dump(findings, fh, indent=2)
print("findings.json saved →", [f["smell"] for f in findings])
cost_report()

---
# Part 2 · 🤝 The Refactoring Team

**⏱ ~60 min**

In Part 1 one agent *found* problems. Now we build a **cooperating team** that *fixes* them — and, crucially,
**proves** the fix is safe before anyone accepts it.

```
        ┌──────────┐   findings   ┌────────────┐  candidate  ┌────────┐
   ───▶ │ Auditor  │ ───────────▶ │ Refactorer │ ──────────▶ │   QA   │ ──▶ ✅ / ❌
        └──────────┘              └────────────┘             └────────┘
                                        ▲                         │
                                        └──── failure notes ──────┘
```

## 2.1 · How do agents talk? The shared-state ("blackboard") pattern

| Pattern | How it works | Used by |
|---|---|---|
| **Message passing** | agents send addressed messages to each other | AutoGen-style conversations |
| **Shared state (blackboard)** | one state object every agent reads and writes | LangGraph, CrewAI, our team |

Shared state wins for *pipelines* because it is inspectable, serialisable and **replayable** — you can always
answer "why did the system do that?", which is the hardest question in agent debugging.

In [ ]:
from dataclasses import dataclass, field
import datetime

@dataclass
class WorkflowState:
    """The blackboard: one source of truth all agents read/write."""
    source_path: str
    original_code: str = ""
    candidate_code: str = ""      # the refactorer's latest proposal
    findings: list = field(default_factory=list)
    verdict: dict = field(default_factory=dict)
    accepted: bool = False
    iteration: int = 0
    history: list = field(default_factory=list)   # the audit trail

    def record(self, agent: str, event: str, detail: str = ""):
        stamp = datetime.datetime.now().strftime("%H:%M:%S")
        self.history.append(f"[{stamp}] {agent:12} | {event:22} | {detail}")
        print(self.history[-1])

state = WorkflowState(source_path="inventory.py")
state.original_code = open(state.source_path).read()
state.record("system", "state initialised", f"{len(state.original_code)} chars of smelly code")

## 2.2 · Agent 2: the **Refactoring Agent** 🔧

Its contract: *given the source + the auditor's findings, produce a complete rewritten module that fixes the
smells **without changing behaviour***.

Three guardrails make this reliable — read the prompt carefully, every line earns its place:
1. **Explicit API freeze** — name the class and the methods that must survive.
2. **Behaviour freeze** — same returns for the same inputs, spelled out.
3. **A single output format** — one fenced code block, so parsing is mechanical.

In [ ]:
REFACTORER_PROMPT = """You are an expert Python refactoring engineer.
Rewrite the ENTIRE module to fix the reported code smells.

HARD RULES — violating any of these makes your output worthless:
1. Public API must not change: class InventoryManager with methods
   __init__(items=None), add_item, process_order, refund_order, get_stock.
2. Behaviour must be IDENTICAL: same return values for the same inputs,
   including all totals, discounts, tax and status strings.
3. Use ONLY the Python standard library. Do NOT invent helper packages.
4. Replace magic numbers with named module-level constants.
5. Extract duplicated pricing logic into one private helper method.
6. Fix the mutable default argument. Remove dead code.

Output ONLY one fenced python code block containing the complete module.
No explanations before or after."""


def extract_code_block(text: str) -> str:
    """Pull the (last) fenced python block out of an LLM reply."""
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if not blocks:
        raise ValueError("Refactorer produced no code block")
    return blocks[-1].strip() + "\n"


def refactor(state: WorkflowState) -> WorkflowState:
    state.record("refactorer", "thinking", f"{len(state.findings)} findings to fix")
    findings_txt = json.dumps(state.findings, indent=1)
    reply = llm(
        f"MODULE TO REFACTOR:\n```python\n{state.original_code}\n```\n\n"
        f"AUDITOR FINDINGS:\n{findings_txt}\n\nRewrite the full module now.",
        system_prompt=REFACTORER_PROMPT, max_new_tokens=2000, temperature=0.1,
    )
    state.candidate_code = extract_code_block(reply)
    state.record("refactorer", "candidate produced", f"{len(state.candidate_code)} chars")
    return state

print("✅ refactorer ready")

## 2.3 · Agent 3: the **QA Verification Agent** 🛡️

This agent is the *only* reason we can trust anything the refactorer produces. It runs three gates,
**cheapest first** — fail fast, spend compute only on survivors:

| Gate | Cost | Catches |
|---|---|---|
| 1 · `ast.parse` | microseconds | broken syntax, truncated output |
| 2 · `pytest` in a sandbox | ~1 second | **behaviour changes** — the dangerous ones |
| 3 · complexity delta | ~1 second | "refactorings" that made things worse |

> 🔑 **Note what is missing from this agent: an LLM.** The verifier is pure, boring, deterministic code.
> Never let the thing that might hallucinate also be the thing that decides whether it hallucinated.

In [ ]:
import shutil, tempfile, os, sys

def avg_complexity(path: str) -> float:
    """Average cyclomatic complexity of a file (via radon's JSON output)."""
    raw = subprocess.run(["radon", "cc", "-j", path], capture_output=True, text=True).stdout
    data = json.loads(raw or "{}")
    scores = [b["complexity"] for blocks in data.values() for b in blocks]
    return sum(scores) / len(scores) if scores else 0.0


def qa_verify(state: WorkflowState) -> WorkflowState:
    verdict = {"syntax": False, "tests": False, "improved": False, "notes": []}

    # ---- Gate 1: does it even parse? ----
    try:
        ast.parse(state.candidate_code)
        verdict["syntax"] = True
    except SyntaxError as e:
        verdict["notes"].append(f"SyntaxError: {e}")
        state.verdict = verdict
        state.record("qa", "❌ REJECTED at gate 1", str(e)[:60])
        return state

    # ---- Gate 2: behaviour preserved? (run tests in a sandbox) ----
    with tempfile.TemporaryDirectory() as sandbox:
        open(os.path.join(sandbox, "inventory.py"), "w").write(state.candidate_code)
        shutil.copy("test_inventory.py", sandbox)
        r = subprocess.run([sys.executable, "-m", "pytest", "test_inventory.py", "-q"],
                           cwd=sandbox, capture_output=True, text=True, timeout=180)
        verdict["tests"] = (r.returncode == 0)
        if not verdict["tests"]:
            verdict["notes"].append("pytest output:\n" + r.stdout[-800:])
            state.verdict = verdict
            state.record("qa", "❌ REJECTED at gate 2", "tests failed")
            return state

        # ---- Gate 3: is it actually better? ----
        cc_before = avg_complexity(state.source_path)
        cc_after = avg_complexity(os.path.join(sandbox, "inventory.py"))
        verdict["cc_before"], verdict["cc_after"] = round(cc_before, 2), round(cc_after, 2)
        verdict["improved"] = cc_after <= cc_before
        if not verdict["improved"]:
            verdict["notes"].append(f"complexity got WORSE: {cc_before} → {cc_after}")

    state.verdict = verdict
    ok = verdict["syntax"] and verdict["tests"] and verdict["improved"]
    state.record("qa", "✅ ACCEPTED" if ok else "❌ REJECTED at gate 3",
                 f"CC {verdict.get('cc_before','?')} → {verdict.get('cc_after','?')}")
    return state

print("✅ QA gates ready — no LLM inside")

## 2.4 · The **Orchestrator**: closing the loop 🔁

The orchestrator adds the two properties every agentic system needs:

- **A retry loop** — if QA rejects, the refactorer gets another shot **with the failure notes fed back**.
  That feedback is what makes iteration 2 smarter than iteration 1.
- **A hard stop** — `max_iterations`. Without it, a confused agent burns your budget forever.

In [ ]:
def run_workflow(source_path: str, max_iterations: int = 3) -> WorkflowState:
    state = WorkflowState(source_path=source_path)
    state.original_code = open(source_path).read()

    state.record("orchestrator", "▶ audit phase")
    state.findings = audit(source_path)
    state.record("auditor", "findings", ", ".join(f["smell"] for f in state.findings)[:80])

    feedback = ""
    for state.iteration in range(1, max_iterations + 1):
        state.record("orchestrator", f"▶ iteration {state.iteration}/{max_iterations}")

        if feedback:   # feed QA failure notes back to the refactorer
            state.findings = state.findings + [
                {"smell": "PREVIOUS ATTEMPT FAILED QA", "why": feedback[:400],
                 "fix": "produce a corrected complete module"}]

        state = refactor(state)
        state = qa_verify(state)

        v = state.verdict
        if v.get("syntax") and v.get("tests") and v.get("improved"):
            state.accepted = True
            break
        feedback = " | ".join(v.get("notes", []))[:400]

    state.record("orchestrator",
                 "🏁 DONE — patch " + ("ACCEPTED" if state.accepted else "NOT accepted"),
                 f"after {state.iteration} iteration(s)")
    return state


state = run_workflow("inventory.py", max_iterations=3)
cost_report()

### 🔍 Inspect the result

> ⚠️ **It is fine (and instructive) if an iteration was rejected.** Models *do* break behaviour — and your QA
> gate just caught it in front of your eyes. That catch **is** the lesson: the gate, not the model, is what
> makes the system trustworthy.

In [ ]:
import difflib

if state.accepted:
    diff = difflib.unified_diff(
        state.original_code.splitlines(keepends=True),
        state.candidate_code.splitlines(keepends=True),
        fromfile="inventory.py (before)", tofile="inventory.py (after)")
    print("".join(diff))
else:
    print("No accepted patch — inspect state.verdict['notes'] to see why:")
    print("\n".join(state.verdict.get("notes", []))[:1200])

In [ ]:
# Before / after scoreboard
mi = lambda p: subprocess.run(["radon", "mi", "-s", p], capture_output=True, text=True).stdout.strip()
print("METRIC                BEFORE                       AFTER")
print("-" * 70)
if state.accepted:
    open("inventory_refactored.py", "w").write(state.candidate_code)
    print(f"avg complexity        {state.verdict['cc_before']:<28} {state.verdict['cc_after']}")
    print(f"maintainability       {mi('inventory.py'):<28.28} {mi('inventory_refactored.py'):<.28}")
    print(f"tests                 7 passed                     7 passed  ✅ behaviour preserved")
else:
    print("(accept a patch first)")

In [ ]:
# The audit trail: every decision, replayable. THIS is how you debug agents.
print("\n".join(state.history))

## 2.5 · 🔥 The Failure-Pattern Lab

Time to break things on purpose. Agentic systems fail in *recurring, nameable* ways. Knowing the names is half the defence.

| # | Failure pattern | What it looks like | Mitigation (which we built) |
|---|---|---|---|
| 1 | **Plausible-but-wrong code** | patch looks perfect, silently changes a constant | gate 2 — behaviour tests |
| 2 | **Contract drift** | model returns prose instead of a code block | strict parser + retry |
| 3 | **Reward hacking** | "simplifies" by deleting features | gate 2 + gate 3 together |
| 4 | **Runaway loops** | agent retries forever | `max_iterations` |
| 5 | **Unverifiable claims** | "I refactored it safely!" | no LLM inside the verifier |

### The sabotage demo

Below we hand QA a patch where a single VAT constant was changed from `0.25` to `0.20`. It parses. It looks
clean. It is a €-losing production bug.

In [ ]:
sabotage = state.original_code.replace("total * 0.25", "total * 0.20")  # 🕵️ subtle!

evil_state = WorkflowState(source_path="inventory.py")
evil_state.original_code = state.original_code
evil_state.candidate_code = sabotage
evil_state = qa_verify(evil_state)

print("\nQA verdict on the sabotaged patch:")
print(json.dumps({k: v for k, v in evil_state.verdict.items() if k != "notes"}, indent=2))
print("\nFailure notes:\n", "\n".join(evil_state.verdict["notes"])[:600])

☝️ The tests caught in ~1 second what code review would probably miss. Now flip it around:

> 💬 **Discuss (2 min):** gate 2 is only as strong as the test suite. What sabotage would slip through *our*
> seven tests? *(Hint: is `add_item`'s discount parameter tested at all? What about float edge cases?)*
> This is exactly why **test debt is the most expensive debt** in an agentic pipeline — it silently lowers the
> ceiling on everything an agent is allowed to do.

## 2.6 · ✍️ Exercise 2 (15 min) — pick one

**A · The Documenter agent.** Add a 4th agent that adds Google-style docstrings to the accepted module *without
changing a single identifier or expression* — then push its output through `qa_verify` like everyone else.

**B · A 4th gate.** Extend `qa_verify` so the maintainability index must also improve, not just complexity.
Does the pipeline still accept anything? What does that tell you about setting quality bars?

**C · Break the refactorer.** Delete one HARD RULE from `REFACTORER_PROMPT` and re-run. Which rule was
load-bearing?

In [ ]:
# 🖊️ Your Exercise 2 workspace

DOCSTRING_PROMPT = """You are a documentation engineer. Add concise Google-style
docstrings to every class and method. Change NOTHING else — not one identifier,
not one expression. Output ONLY one fenced python code block."""

def document(state):
    # TODO: call llm() with DOCSTRING_PROMPT + state.candidate_code,
    #       extract_code_block(), put the result back into state.candidate_code,
    #       then re-run qa_verify(state) and keep it ONLY if accepted.
    pass

<details><summary>💡 Solution sketch (option A)</summary>

```python
def document(state):
    reply = llm(f"```python\n{state.candidate_code}\n```",
                system_prompt=DOCSTRING_PROMPT, max_new_tokens=2000, temperature=0.1)
    proposal = extract_code_block(reply)
    trial = WorkflowState(source_path=state.source_path)
    trial.original_code = state.original_code
    trial.candidate_code = proposal
    trial = qa_verify(trial)
    if trial.verdict.get("tests"):          # docstrings must not change behaviour
        state.candidate_code = proposal
        state.record("documenter", "✅ docstrings accepted")
    else:
        state.record("documenter", "❌ rejected — behaviour changed")
    return state

state = document(state)
```
Note the pattern: **a new agent does not get a new trust level.** It goes through the same gate.
</details>

---
# Part 3 · 💳 Technical Debt: Classify, Triage, Resolve

**⏱ ~40 min**

Parts 1–2 operated on **code**. But most technical debt is first reported in **words** — issue trackers, PR
comments, TODO notes. In this part the team learns to read the backlog, price it, and act on it.

**Interest vs principal** — the metaphor that makes debt a management conversation instead of an engineering complaint:
- **principal** = what the proper fix costs, once.
- **interest** = what the shortcut costs you, *every sprint until then*.

## 3.1 · A slice of a real-world issue tracker

Ten issues of the kind **BEACon-TD** (JSS 2025) was trained on. Each hides a debt type — sometimes several.
We keep hand labels so we can *score* our classifier instead of admiring it.

In [ ]:
ISSUES = [
 {"id": 101, "text": "The OrderService class is 3000 lines and does payment, shipping AND emails. Every change breaks something unrelated. We need to split it before adding new payment providers.", "gold": "design"},
 {"id": 102, "text": "There are zero tests for the refund flow. We only find regressions when customers complain. Adding tests keeps getting postponed for feature work.", "gold": "test"},
 {"id": 103, "text": "The README still describes the v1 API. New joiners lose days because the setup guide is wrong and the architecture diagram shows services we deleted last year.", "gold": "documentation"},
 {"id": 104, "text": "We're pinned to Django 2.2 which reached end-of-life. Security patches no longer land and two dependencies refuse to install alongside it.", "gold": "dependency"},
 {"id": 105, "text": "TODO left from the March crunch: error handling in the CSV importer just swallows exceptions with a bare except and returns None. Works until it doesn't.", "gold": "defect"},
 {"id": 106, "text": "The nightly ETL takes 6 hours because it re-reads the entire table every run. A watermark column was proposed in 2023 but never implemented.", "gold": "design"},
 {"id": 107, "text": "Deployment is a 14-step manual runbook involving three people and an SSH session. One typo in step 9 took prod down last month. We need CI/CD.", "gold": "build"},
 {"id": 108, "text": "Variable names in the pricing module are a, b, tmp2 and data_final_v3. Code review of any pricing change takes twice as long as it should.", "gold": "code"},
 {"id": 109, "text": "Our fork of the auth library diverged 200 commits from upstream. Merging upstream security fixes now takes a full sprint each time.", "gold": "dependency"},
 {"id": 110, "text": "The ML model in production was trained on 2022 data and nobody saved the training script or the random seed. Retraining reproducibly is currently impossible.", "gold": "code"},
]
print(f"{len(ISSUES)} issues loaded")

## 3.2 · Agent 4: the **TD Classifier**

BEACon-TD's production answer is a *fine-tuned transformer* — small, fast, cheap, consistent. Today we
approximate the task with a **constrained-label prompt** (zero-shot classification).

Note the constraint pattern: give the model a *closed* label set, demand one word, and **snap the answer back
into the set** in code. Never trust free text where an enum belongs.

In [ ]:
LABELS = ["design", "code", "test", "documentation",
          "dependency", "build", "defect", "requirement"]

CLASSIFIER_PROMPT = f"""You classify technical-debt reports from issue trackers.
Allowed labels (choose EXACTLY one): {", ".join(LABELS)}.

Definitions:
- design: architectural problems, god classes, wrong abstractions, inefficient designs
- code: poor readability/naming, code smells, missing reproducibility of code artifacts
- test: missing/weak/flaky tests, low coverage
- documentation: missing or outdated docs/diagrams/guides
- dependency: outdated/EOL libraries, diverged forks, version conflicts
- build: manual/fragile build, deployment or CI/CD problems
- defect: known bugs or error-handling gaps deliberately left in the code
- requirement: implementation diverged from what was actually required

Reply with ONLY the label word. Nothing else."""


def classify_issue(text: str) -> str:
    reply = llm(f"ISSUE:\n{text}\n\nLabel:", system_prompt=CLASSIFIER_PROMPT,
                max_new_tokens=8, temperature=0.0)
    reply = reply.lower().strip().split()[0].strip(".,:")
    return reply if reply in LABELS else "code"   # snap to label set


import pandas as pd
rows = []
for issue in ISSUES:
    pred = classify_issue(issue["text"])
    rows.append({"id": issue["id"], "gold": issue["gold"], "predicted": pred,
                 "✓": "✅" if pred == issue["gold"] else "❌",
                 "issue": issue["text"][:70] + "…"})
df = pd.DataFrame(rows)
accuracy = (df["gold"] == df["predicted"]).mean()
print(f"Zero-shot accuracy: {accuracy:.0%}\n")
df

### 🤔 Reading the errors

Look at the ❌ rows. Most misclassifications are **genuinely ambiguous** — issue 110 (unreproducible ML
training) is arguably `code`, `design`, *and* a process problem. Real issues carry several debt types at once,
which is why BEACon-TD treats this as **multi-label** classification with 13 types.

> 💬 **Discuss (2 min):** your accuracy here is maybe 60–90%. A fine-tuned BEACon-TD model does better,
> costs ~1000× less per issue, runs on-prem, and — crucially — gives the *same* answer tomorrow.
> When is prompting the right call, and when do you fine-tune?

## 3.3 · Agent 5: the **Triage Agent** — pricing the debt

In [ ]:
TRIAGE_PROMPT = """You are an engineering manager triaging technical debt.
For the issue, estimate:
- interest: recurring pain per sprint if NOT fixed, integer 1 (minor) to 5 (severe)
- principal: effort of the proper fix, integer 1 (hours) to 5 (multi-sprint)
- rationale: one short sentence

Reply ONLY with a JSON object: {"interest": int, "principal": int, "rationale": str}"""


def triage(issue) -> dict:
    reply = llm(f"ISSUE:\n{issue['text']}", system_prompt=TRIAGE_PROMPT,
                max_new_tokens=160, temperature=0.1, json_mode=True)
    try:
        score = json.loads(reply)
    except json.JSONDecodeError:
        score = {"interest": 3, "principal": 3, "rationale": "parse failed"}
    # priority = pain per unit of effort
    score["priority"] = round(score.get("interest", 3) / max(score.get("principal", 3), 1), 2)
    return score


backlog = []
for issue in ISSUES:
    s = triage(issue)
    backlog.append({"id": issue["id"], "type": classify_issue(issue["text"]),
                    "interest": s.get("interest"), "principal": s.get("principal"),
                    "priority": s["priority"], "rationale": s.get("rationale", "")[:60]})

pd.DataFrame(sorted(backlog, key=lambda r: -r["priority"]))

> 💬 **Discuss (2 min):** the LLM just made **resource-allocation judgements**. Would you let this ranking
> drive sprint planning directly? What human checkpoint would you insert, and *why exactly there*?
> *(No consensus answer — the trade-off between automation speed and accountable judgement is the point.)*

## 3.4 · 🏆 The capstone: the full pipeline

Everything you built, in one function: classify → triage → audit → refactor ⇄ QA → **report**.

In [ ]:
def full_pipeline(code_path: str, issues: list, max_iterations: int = 3) -> str:
    """Issues + code in → verified fix + Markdown debt report out."""
    log = lambda *a: print("  ", *a)

    print("① Classifying & triaging the issue backlog …")
    ranked = []
    for it in issues:
        s = triage(it)
        ranked.append({**it, "type": classify_issue(it["text"]), **s})
    ranked.sort(key=lambda r: -r["priority"])

    print("② Auditing the code …")
    code_findings = audit(code_path)
    log(f"{len(code_findings)} findings:", ", ".join(f.get("smell", "?") for f in code_findings)[:80])

    print("③ Refactor ⇄ QA loop …")
    st = WorkflowState(source_path=code_path)
    st.original_code = open(code_path).read()
    st.findings = code_findings
    feedback = ""
    for st.iteration in range(1, max_iterations + 1):
        if feedback:
            st.findings = code_findings + [{"smell": "PREVIOUS ATTEMPT FAILED QA",
                                            "why": feedback[:400],
                                            "fix": "produce a corrected complete module"}]
        try:
            st = refactor(st)
        except ValueError as e:
            feedback = str(e); log(f"iter {st.iteration}: ❌ {e}"); continue
        st = qa_verify(st)
        v = st.verdict
        if v.get("syntax") and v.get("tests") and v.get("improved"):
            st.accepted = True
            log(f"iter {st.iteration}: ✅ accepted")
            break
        feedback = " | ".join(v.get("notes", []))[:400]
        log(f"iter {st.iteration}: ❌ {feedback[:70]}")

    print("④ Writing TECH_DEBT_REPORT.md …")
    lines = ["# Technical Debt Report", "",
             f"*Generated by a cooperative LLM-agent pipeline · model: {MODEL_NAME}*", "",
             "## 1 · Prioritised issue backlog (top 5)", "",
             "| Rank | Issue | Type | Interest | Principal | Priority |", "|--|--|--|--|--|--|"]
    for rank, r in enumerate(ranked[:5], 1):
        lines.append(f"| {rank} | #{r['id']} {r['text'][:55]}… | {r['type']} | "
                     f"{r['interest']} | {r['principal']} | {r['priority']} |")

    lines += ["", "## 2 · Code audit findings", ""]
    for f in code_findings:
        lines.append(f"- **{f.get('smell','?')}** ({f.get('severity','?')}, {f.get('location','?')}): "
                     f"{f.get('why','')} → *{f.get('fix','')}*")

    lines += ["", "## 3 · Automated refactoring outcome", ""]
    if st.accepted:
        open("inventory_refactored.py", "w").write(st.candidate_code)
        lines += [f"- ✅ Patch **accepted**: all 7 behaviour tests pass; "
                  f"avg complexity {st.verdict['cc_before']} → {st.verdict['cc_after']}",
                  "- Verified code saved to `inventory_refactored.py`"]
    else:
        lines += [f"- ❌ No patch met the QA bar within {max_iterations} iterations "
                  f"(last reason: {feedback[:120]}). Original code kept — "
                  "**the gate held; nothing unverified shipped.**"]

    print("⑤ Scanning ML code with MLScent …")
    ml_report = run_mlscent("ml_project")
    counts = ml_report.split("Smell Counts:")[-1].strip().splitlines()[:8]
    lines += ["", "## 4 · ML-specific smells (`ml_project/` · MLScent)", ""]
    lines += [f"- {c.strip()}" for c in counts if c.strip()]
    lines += ["", "*Full MLScent report: `output/analysis_report.txt` — each finding includes a How-to-fix.*"]

    report = "\n".join(lines)
    open("TECH_DEBT_REPORT.md", "w").write(report)
    return report


report = full_pipeline("inventory.py", ISSUES, max_iterations=3)
cost_report()

In [ ]:
from IPython.display import Markdown, display
display(Markdown(report))

🏆 **You built an end-to-end technical-debt management pipeline** — perception (tools), reasoning (LLM),
action (refactoring), verification (gates), and reporting. The **architecture** is what made it trustworthy,
not the size of the model.

## 3.5 · ✍️ Final challenge — bring your own code

Paste any Python module of yours into the cell below, **write 2–3 behaviour tests for it**, point
`qa_verify`'s sandbox at your test file, and run `full_pipeline` on it.

- No tests you trust? → notice how uncomfortable that feels. **That discomfort is test debt, quantified emotionally.**
- Model mangles your code? → the gate rejects it. Working as designed.
- Everything passes first try? → raise the bar in `qa_verify` (make MI improvement mandatory) and see what happens.

In [ ]:
%%writefile my_module.py
# 🖊️ Paste YOUR code here, then adapt test_inventory.py-style tests for it,
# point qa_verify's sandbox at your test file, and run:
#   report = full_pipeline("my_module.py", ISSUES)

def example(a, b):
    return a + b

---
# Part 4 · 🚀 Production Frameworks (bonus / take-home)

**⏱ ~50 min, self-paced**

Parts 1–3 built every pattern **by hand**, on purpose, so nothing is magic. Part 4 rebuilds the *same*
pipeline with the production stack:

| Section | Tool | What it buys you |
|---|---|---|
| 4.1 | **LangGraph** | the loop as a typed `StateGraph` — free diagram, streaming, checkpointing |
| 4.2 | **Deep Agents** | autonomous planning (`write_todos`), file tools, real tool-calling |
| 4.3 | **`debtbuster`** | the whole thing packaged as a `pip install`-able CLI that exits non-zero in CI |

> 🔑 **Watch what does NOT change:** the QA gates. They are byte-for-byte the same across all three versions,
> because **verification is a property of the task, not of the framework**.

## 4.1 · The refactoring team, rebuilt in **LangGraph**

LangGraph models a workflow as **typed state + a graph of nodes**. Our `WorkflowState` becomes a `TypedDict`,
our three agents become nodes, and the retry loop becomes a **conditional edge**.

Note we keep calling our own `llm()` — no extra LangChain provider package needed for this section.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

AUDITOR_SYS = ("You are a strict senior code reviewer. Given a Python module, list its "
               "worst code smells as short bullet points (max 6). Be specific: name the "
               "function and the problem. No fixes yet, no prose intro.")
REFACTORER_SYS = ("You are a careful refactoring engineer. Rewrite the module fixing the "
                  "listed smells. HARD INVARIANTS: same public API and behaviour; stdlib only; "
                  "replace magic numbers with named constants; remove duplication and dead code. "
                  "Reply with ONE ```python``` block containing the FULL module, nothing else.")


def qa_gates(original_path: str, candidate: str) -> dict:
    """The Part-2 gates, in the shape LangGraph wants. Still no LLM inside."""
    try:
        ast.parse(candidate)                                        # gate 1
    except SyntaxError as e:
        return {"ok": False, "why": f"gate 1 (syntax): {e}"}
    with tempfile.TemporaryDirectory() as tmp:                      # gate 2
        pathlib.Path(tmp, "inventory.py").write_text(candidate)
        shutil.copy("test_inventory.py", tmp)
        r = subprocess.run([sys.executable, "-m", "pytest", "-q", "test_inventory.py"],
                           cwd=tmp, capture_output=True, text=True, timeout=180)
        if r.returncode != 0:
            return {"ok": False, "why": "gate 2 (behaviour): " + r.stdout[-300:]}
        cc_after = avg_complexity(str(pathlib.Path(tmp, "inventory.py")))
    cc_before = avg_complexity(original_path)                       # gate 3
    if cc_after > cc_before:
        return {"ok": False, "why": f"gate 3 (quality): CC worsened {cc_before:.2f} → {cc_after:.2f}"}
    return {"ok": True, "why": f"all gates passed · CC {cc_before:.2f} → {cc_after:.2f}"}


class TeamState(TypedDict):
    source: str
    findings: str
    candidate: str
    verdict: str
    accepted: bool
    iteration: int


def auditor_node(state: TeamState) -> dict:
    print("🕵️ auditor …")
    return {"findings": llm(state["source"], system_prompt=AUDITOR_SYS,
                            max_new_tokens=600, temperature=0.1)}


def refactorer_node(state: TeamState) -> dict:
    print(f"🔧 refactorer (iteration {state['iteration'] + 1}) …")
    fb = f"\nPREVIOUS ATTEMPT REJECTED: {state['verdict']}" if state["verdict"] else ""
    reply = llm(f"MODULE:\n```python\n{state['source']}\n```\nSMELLS:\n{state['findings']}{fb}",
                system_prompt=REFACTORER_SYS, max_new_tokens=2000, temperature=0.1)
    try:
        return {"candidate": extract_code_block(reply), "iteration": state["iteration"] + 1}
    except ValueError as e:
        return {"candidate": "", "verdict": str(e), "iteration": state["iteration"] + 1}


def qa_node(state: TeamState) -> dict:
    print("🛡️ qa gates …")
    if not state["candidate"]:
        return {"accepted": False}
    v = qa_gates("inventory.py", state["candidate"])
    print("   ", "✅" if v["ok"] else "❌", v["why"][:90])
    return {"accepted": v["ok"], "verdict": v["why"]}


def route_after_qa(state: TeamState) -> str:
    if state["accepted"]:
        return "done"
    return "give_up" if state["iteration"] >= 3 else "retry"


g = StateGraph(TeamState)
g.add_node("auditor", auditor_node)
g.add_node("refactorer", refactorer_node)
g.add_node("qa", qa_node)
g.add_edge(START, "auditor")
g.add_edge("auditor", "refactorer")
g.add_edge("refactorer", "qa")
g.add_conditional_edges("qa", route_after_qa, {"done": END, "retry": "refactorer", "give_up": END})
team = g.compile()
print("✅ graph compiled")

In [ ]:
# LangGraph draws its own architecture diagram — compare it to the ASCII sketch in Part 2!
from IPython.display import Image, display
try:
    display(Image(team.get_graph().draw_mermaid_png()))
except Exception:
    print(team.get_graph().draw_mermaid())   # fallback: raw mermaid text

In [ ]:
result = team.invoke({"source": open("inventory.py").read(),
                      "findings": "", "candidate": "", "verdict": "",
                      "accepted": False, "iteration": 0})

print("\n" + "=" * 60)
if result["accepted"]:
    open("inventory_langgraph.py", "w").write(result["candidate"])
    print(f"✅ ACCEPTED after {result['iteration']} iteration(s) — {result['verdict']}")
    print("→ saved to inventory_langgraph.py")
else:
    print(f"❌ REJECTED after {result['iteration']} iteration(s) — {result['verdict'][:200]}")
    print("(the gate held; nothing unverified shipped)")
cost_report()

> 💬 **Compare (2 min):** put Part 2's `run_workflow()` next to this graph. Same auditor, same gates, same loop.
> **What did LangGraph buy you?** The diagram for free, typed state, streaming/checkpointing when you need it,
> and a shape your colleagues already recognise. **What did it cost?** A dependency, an abstraction layer, and
> a debugging story that now runs through someone else's code.

## 4.2 · An autonomous **Deep Agent** 🤖

Everything so far had a *fixed* control flow: we decided the order. A **deep agent** decides for itself — it
plans with `write_todos`, calls tools in whatever order it judges useful, reads and writes files, and stops
when it thinks it is done.

`deepagents` talks to models through LangChain, so this one section needs the OpenAI binding package:

In [ ]:
%pip install -q langchain-openai
print("✅ deepagents can now reach OpenAI")

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend


# ⚠️ Two guards every autonomous-agent tool needs. Learned the hard way while building
#    this notebook: the first version let the agent call radon_report("/") — it walked the
#    ENTIRE filesystem and returned a 6.8 MB "tool result". Unbounded tools are a failure
#    pattern, not an edge case.
def _safe(path: str) -> str:
    """Guard 1 — keep the agent inside the workshop directory."""
    p = pathlib.Path(path)
    p = (p if p.is_absolute() else WORKDIR / p).resolve()
    return str(p) if p == WORKDIR or WORKDIR in p.parents else str(WORKDIR)


LIMIT = 3000                       # Guard 2 — every tool result is truncated


def radon_report(path: str) -> str:
    """Cyclomatic-complexity report for a Python file or directory."""
    out = subprocess.run(["radon", "cc", "-s", _safe(path)],
                         capture_output=True, text=True, timeout=120).stdout
    return out[:LIMIT] or "n/a"


def pyexamine_report(path: str) -> str:
    """PyExamine (MSR 2025): 49-metric code-smell report for a directory."""
    subprocess.run(["analyze_code_quality", _safe(path), "--type", "code", "--output", "pyx"],
                   capture_output=True, text=True, timeout=600)
    p = pathlib.Path("pyx.txt")
    return p.read_text()[:LIMIT] if p.exists() else "no report produced"


def mlscent_report(path: str) -> str:
    """MLScent (CAIN 2025): 76 ML-specific anti-pattern detectors for a directory."""
    subprocess.run(["ml_smell_detector", "analyze", _safe(path)],
                   capture_output=True, text=True, timeout=600)
    p = pathlib.Path("output/analysis_report.txt")
    return p.read_text()[:LIMIT] if p.exists() else "no report produced"


deep_auditor = create_deep_agent(
    model=f"openai:{MODEL_NAME}",
    tools=[radon_report, pyexamine_report, mlscent_report],
    backend=FilesystemBackend(root_dir=str(WORKDIR)),   # real files, sandboxed to our workdir
    system_prompt=(
        "You are an autonomous code-quality auditor. Plan your work with write_todos first. "
        "Your filesystem root IS the working directory: the files are 'inventory.py' and "
        "'ml_project/'. Do not go looking for them elsewhere — no /workspace, /root, /app. "
        "Audit ONLY those two, and never pass '/' or any path outside them to a tool. "
        "Use radon_report and pyexamine_report on Python "
        "business code; use mlscent_report only if the code imports ML libraries. Read files "
        "before judging them. Work quickly: at most 8 tool calls in total. "
        "Finish by writing a concise AUDIT.md (max 25 lines) with your top findings, "
        "each with file, smell, and a one-line fix."
    ),
)
print("✅ deep agent ready — tools:", [t.__name__ for t in (radon_report, pyexamine_report, mlscent_report)])

In [ ]:
# Let it loose. Watch the todo list appear, then the tool calls.
run = deep_auditor.invoke(
    {"messages": [("user", "Audit the Python code in this directory (start with inventory.py) "
                           "and produce AUDIT.md.")]},
    config={"recursion_limit": 40},
)

for m in run["messages"]:
    for tc in (getattr(m, "tool_calls", None) or []):
        print(f"🔩 tool call → {tc['name']}({str(tc['args'])[:60]})")

final = run["messages"][-1].content          # reasoning models return a list of blocks
if isinstance(final, list):
    final = "\n".join(b.get("text", "") for b in final
                      if isinstance(b, dict) and b.get("type") == "text")
print("\n--- final answer ---\n", str(final)[:800])

if pathlib.Path("AUDIT.md").exists():
    print("\n📄 AUDIT.md written:\n", pathlib.Path("AUDIT.md").read_text()[:900])
cost_report()

> ⚠️ **Reality check — say this out loud:** an autonomous agent with file-write access and no gate is exactly
> the configuration you should *not* ship. Notice that nothing in section 4.2 ran `pytest`. The deep agent is
> excellent at **exploration and reporting**; Parts 1–3 are what you wrap around it before it is allowed to
> **change** anything.
>
> 🐛 **A real bug from building this notebook.** The first version of `radon_report` had no path guard and
> no length limit. The agent decided to audit `/` — and returned a **6.8 MB** tool result, stalling the run and
> burning the token budget. A fixed-flow pipeline cannot do this, because *you* choose the arguments. The moment
> an agent chooses its own, **every tool needs a bounded input and a bounded output.** That is the tax autonomy
> charges, and it is why `_safe()` and `LIMIT` exist two cells up.

## 4.3 · Package it: `debtbuster`, a lean CLI harness 📦

Notebooks are for learning; **CI runs commands**. Here is the whole pipeline as five small source files, a
config, its own pytest suite, and one entry point that **exits non-zero when the gate rejects** — so a
pipeline fails safely.

In [ ]:
!mkdir -p debtbuster/src/debtbuster debtbuster/tests

In [ ]:
%%writefile debtbuster/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "debtbuster"
version = "0.2.0"
description = "Lean LLM-agent harness: audit, refactor & gate Python code (LLMA4SE Workshop 4)"
requires-python = ">=3.10"
dependencies = [
  "langgraph", "openai",
  "radon", "pylint", "code-quality-analyzer", "pytest",
]

[project.scripts]
debtbuster = "debtbuster.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

In [ ]:
%%writefile debtbuster/src/debtbuster/config.py
"""Single source of truth -- change the brain here, nowhere else."""
MODEL = "gpt-4.1-mini"
MAX_ITERATIONS = 3
TEMPERATURE = 0.1
MAX_TOKENS = 2000

In [ ]:
%%writefile debtbuster/src/debtbuster/tools.py
"""Deterministic eyes. No LLM in this file."""
import json, pathlib, subprocess, sys

_BIN = pathlib.Path(sys.executable).parent

def _exe(name: str) -> str:
    """Prefer the tool installed next to THIS interpreter, else trust PATH.

    Without this, `debtbuster` installed into a venv that is not on PATH cannot
    find its own dependencies -- the console script runs, radon does not.
    """
    candidate = _BIN / name
    return str(candidate) if candidate.exists() else name

def radon_avg_cc(path: str) -> float:
    out = subprocess.run([_exe("radon"), "cc", "-s", "-j", path],
                         capture_output=True, text=True).stdout
    data = json.loads(out or "{}")
    scores = [b["complexity"] for blocks in data.values() for b in blocks]
    return round(sum(scores) / len(scores), 2) if scores else 0.0

def pyexamine(path: str) -> str:
    subprocess.run([_exe("analyze_code_quality"), path, "--type", "code", "--output", "pyx"],
                   capture_output=True, text=True, timeout=600)
    rpt = pathlib.Path("pyx.txt")
    return rpt.read_text()[:3000] if rpt.exists() else ""

In [ ]:
%%writefile debtbuster/src/debtbuster/gates.py
"""The QA gate. Deliberately boring, deliberately LLM-free."""
import ast, pathlib, shutil, subprocess, sys, tempfile
from .tools import radon_avg_cc

def verify(original: str, candidate: str, test_file: str | None) -> dict:
    try:
        ast.parse(candidate)                                        # gate 1: syntax
    except SyntaxError as e:
        return {"ok": False, "why": f"gate 1 (syntax): {e}"}
    if test_file:                                                   # gate 2: behaviour
        with tempfile.TemporaryDirectory() as tmp:
            tgt = pathlib.Path(tmp, pathlib.Path(original).name)
            tgt.write_text(candidate)
            shutil.copy(test_file, tmp)
            r = subprocess.run([sys.executable, "-m", "pytest", "-q", pathlib.Path(test_file).name],
                               cwd=tmp, capture_output=True, text=True, timeout=180)
            if r.returncode != 0:
                return {"ok": False, "why": "gate 2 (behaviour): " + r.stdout[-300:]}
    with tempfile.TemporaryDirectory() as tmp:                      # gate 3: quality
        p = pathlib.Path(tmp, "cand.py"); p.write_text(candidate)
        before, after = radon_avg_cc(original), radon_avg_cc(str(p))
    if after > before:
        return {"ok": False, "why": f"gate 3 (quality): CC {before} -> {after}"}
    return {"ok": True, "why": f"gates passed - CC {before} -> {after}"}

In [ ]:
%%writefile debtbuster/src/debtbuster/brain.py
"""The only file that talks to a model. Swap providers here, nowhere else."""
from openai import OpenAI
from . import config

_client = None

def chat(system: str, user: str) -> str:
    global _client
    if _client is None:
        _client = OpenAI()          # reads OPENAI_API_KEY
    resp = _client.chat.completions.create(
        model=config.MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        temperature=config.TEMPERATURE,
        max_tokens=config.MAX_TOKENS,
    )
    return (resp.choices[0].message.content or "").strip()

In [ ]:
%%writefile debtbuster/src/debtbuster/graph.py
"""The LangGraph team: auditor -> refactorer <-> gates. Mirrors the notebook 1:1."""
import pathlib, re
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from . import config
from .brain import chat
from .gates import verify
from .tools import pyexamine

AUDITOR_SYS = ("You are a strict senior code reviewer. List the module's worst code "
               "smells as max 6 short bullets. Name function + problem. No fixes, no intro.")
REFACTORER_SYS = ("You are a careful refactoring engineer. Rewrite the module fixing the "
                  "listed smells. HARD INVARIANTS: same public API and behaviour; stdlib only; "
                  "named constants for magic numbers; remove duplication and dead code. "
                  "Reply with ONE ```python``` block with the FULL module, nothing else.")

class State(TypedDict):
    path: str
    test_file: str | None
    source: str
    findings: str
    candidate: str
    verdict: str
    accepted: bool
    iteration: int

def _extract(text: str) -> str:
    m = re.findall(r"```(?:python)?\s*(.*?)```", text, re.S)
    if not m:
        raise ValueError("no code block in reply")
    return m[-1].strip() + "\n"

def auditor(state: State) -> dict:
    evidence = pyexamine(str(pathlib.Path(state["path"]).parent))
    return {"findings": chat(AUDITOR_SYS,
                             f"STATIC EVIDENCE:\n{evidence}\n\nMODULE:\n{state['source']}")}

def refactorer(state: State) -> dict:
    fb = f"\nPREVIOUS ATTEMPT REJECTED: {state['verdict']}" if state["verdict"] else ""
    reply = chat(REFACTORER_SYS,
                 f"MODULE:\n```python\n{state['source']}\n```\nSMELLS:\n{state['findings']}{fb}")
    try:
        return {"candidate": _extract(reply), "iteration": state["iteration"] + 1}
    except ValueError as e:
        return {"candidate": "", "verdict": str(e), "iteration": state["iteration"] + 1}

def qa(state: State) -> dict:
    if not state["candidate"]:
        return {"accepted": False}
    v = verify(state["path"], state["candidate"], state["test_file"])
    return {"accepted": v["ok"], "verdict": v["why"]}

def _route(state: State) -> str:
    if state["accepted"]:
        return "done"
    return "give_up" if state["iteration"] >= config.MAX_ITERATIONS else "retry"

def build():
    g = StateGraph(State)
    g.add_node("auditor", auditor); g.add_node("refactorer", refactorer); g.add_node("qa", qa)
    g.add_edge(START, "auditor"); g.add_edge("auditor", "refactorer"); g.add_edge("refactorer", "qa")
    g.add_conditional_edges("qa", _route, {"done": END, "retry": "refactorer", "give_up": END})
    return g.compile()

In [ ]:
%%writefile debtbuster/src/debtbuster/cli.py
"""debtbuster: audit | fix -- the whole UX in ~40 lines."""
import argparse, pathlib, sys
from .graph import build
from .tools import radon_avg_cc, pyexamine

def main() -> int:
    ap = argparse.ArgumentParser(prog="debtbuster",
                                 description="LLM-agent code auditing & gated refactoring")
    sub = ap.add_subparsers(dest="cmd", required=True)
    a = sub.add_parser("audit", help="static + agent audit of a file")
    a.add_argument("path")
    f = sub.add_parser("fix", help="run the auditor->refactorer<->QA team on a file")
    f.add_argument("path")
    f.add_argument("--tests", default=None, help="pytest file pinning behaviour (enables gate 2)")
    args = ap.parse_args()

    src = pathlib.Path(args.path).read_text()
    if args.cmd == "audit":
        print(f"avg cyclomatic complexity: {radon_avg_cc(args.path)}")
        print(pyexamine(str(pathlib.Path(args.path).parent))[:1500] or "(no PyExamine findings)")
        return 0

    team = build()
    out = team.invoke({"path": args.path, "test_file": args.tests, "source": src,
                       "findings": "", "candidate": "", "verdict": "",
                       "accepted": False, "iteration": 0})
    if out["accepted"]:
        dst = pathlib.Path(args.path).with_suffix(".refactored.py")
        dst.write_text(out["candidate"])
        print(f"ACCEPTED after {out['iteration']} iteration(s): {out['verdict']}\n-> {dst}")
        return 0
    print(f"REJECTED: {out['verdict'][:300]}\n(the gate held - nothing unverified shipped)")
    return 1

if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile debtbuster/src/debtbuster/__init__.py
"""debtbuster -- lean LLM-agent harness from LLMA4SE Workshop 4."""
__version__ = "0.2.0"

In [ ]:
%%writefile debtbuster/tests/test_gates.py
"""The harness tests ITS OWN gates -- the most load-bearing code gets the tests."""
from debtbuster.gates import verify

GOOD = "def f(x):\n    return x + 1\n"
BAD_SYNTAX = "def f(x:\n    return"

def test_gate1_rejects_broken_syntax(tmp_path):
    orig = tmp_path / "m.py"; orig.write_text(GOOD)
    v = verify(str(orig), BAD_SYNTAX, test_file=None)
    assert not v["ok"] and "gate 1" in v["why"]

def test_gates_accept_identical_code(tmp_path):
    orig = tmp_path / "m.py"; orig.write_text(GOOD)
    v = verify(str(orig), GOOD, test_file=None)
    assert v["ok"]

def test_gate3_rejects_complexity_regression(tmp_path):
    orig = tmp_path / "m.py"; orig.write_text(GOOD)
    worse = ("def f(x):\n"
             "    if x > 0:\n"
             "        if x > 1:\n"
             "            if x > 2:\n"
             "                return x\n"
             "    return x + 1\n")
    v = verify(str(orig), worse, test_file=None)
    assert not v["ok"] and "gate 3" in v["why"]

In [ ]:
# Install the harness in editable mode and prove it works — tests first, like adults
%pip install -q -e ./debtbuster
!cd debtbuster && python -m pytest tests/ -q

In [ ]:
# The CLI, on the same patient as Parts 1–3:
!debtbuster audit inventory.py
print("=" * 60)
!debtbuster fix inventory.py --tests test_inventory.py

### Where does `debtbuster` actually live? 🤔

A fair question, since we are in a hosted notebook: **nothing here came from your laptop.** The `%%writefile`
cells above created `debtbuster/` on the **Colab VM's own disk**, and `pip install -e` registered its
`debtbuster` console script on that VM's `PATH`. `!debtbuster …` is then just a shell command in the VM.

Two consequences worth internalising:

- **It is ephemeral.** When the runtime disconnects, the package is gone. That is fine for a workshop and
  fatal for real work — which is why the next cell shows the durable route.
- **We drive it with `!`, not `import debtbuster`.** An editable install is not importable by the
  *already-running* kernel without a restart, but a **subprocess** picks it up immediately. Using the CLI is
  also the honest test: it is how CI will invoke it.

### The durable route: install it from GitHub 🌍

Once the package lives in a Git repository, any Colab, any CI job, any laptop can install the *same* harness
with one line — no `%%writefile`, no copy-paste:

```bash
pip install "git+https://github.com/KarthikShivasankar/LLMA4SE.git#subdirectory=debtbuster"
```

The `#subdirectory=` fragment is the part people miss: it tells pip the `pyproject.toml` is **not** at the repo
root. Same syntax you already used today — it is exactly how MLScent used to be installed.

> 🔑 **The key travels through the environment, not the code.** `debtbuster` builds its OpenAI client with
> `OpenAI()`, which reads `OPENAI_API_KEY` from the environment. Because §0.2 put your key into
> `os.environ`, every `!debtbuster` subprocess **inherits it automatically**. Nothing is baked into the
> package — which is exactly why the same wheel is safe to install in CI, where the key comes from a secret.

In [ ]:
# The GitHub route — the same harness, installed the way CI would install it.
# Uncomment to run it for real (needs the repo pushed with the debtbuster/ directory):

# %pip install -q "git+https://github.com/KarthikShivasankar/LLMA4SE.git#subdirectory=debtbuster"

import os, shutil
print("OPENAI_API_KEY visible to subprocesses:", "OPENAI_API_KEY" in os.environ)
print("debtbuster on PATH                    :", shutil.which("debtbuster") or "not installed")
print("\nBecause the key lives in the environment, this works identically whether debtbuster")
print("was built by %%writefile above or installed from GitHub. Same command, same result:")
print("    debtbuster fix inventory.py --tests test_inventory.py")

📦 **That is the whole harness.** Five source files, a config, its own tests, one CLI —
`pip install`-able into any CI job:

```bash
debtbuster fix src/module.py --tests tests/test_module.py   # exit 1 when the gate rejects
```

## 4.4 · ✍️ Exercise 4 (pick one)

**A · MLScent gate.** Add a `--ml` flag that also runs `ml_smell_detector` and refuses to accept a patch that
*increases* the ML smell count.

**B · Model swap.** Point `config.MODEL` at `gpt-4.1` (or `gpt-4o-mini`). Measure: iterations to acceptance,
wall-clock, and cost. Is the expensive model actually cheaper per accepted patch?

**C · Deep-agent gate.** Give the deep agent the `qa_gates` function as a *tool* and instruct it never to
report a fix it has not gated. Does an autonomous agent voluntarily verify itself?

In [ ]:
# 🖊️ Your Exercise 4 workspace

## 4.5 · Checkpoint ✅

1. Why does the QA gate stay identical across the hand-rolled, LangGraph and deep-agent versions?
2. What is the one-line change that repoints this whole harness at a different model or provider?
3. Which part of today's pipeline would you *never* let run unattended on a production repo, and why?

<details><summary>Answers</summary>

1. Because **verification is a property of the task, not of the framework**. Frameworks orchestrate; they do not decide what "correct" means.
2. `config.MODEL` (or `MODEL_NAME` in the notebook). Everything else routes through one `chat()` / `llm()` function — that is why it was worth centralising on the very first cell.
3. The **write** step. Auditing and reporting are safe to automate; changing code is safe to automate only behind a gate *plus* human review. Agents propose; verified pipelines and humans dispose.
</details>

---
# 🎓 Wrap-up

## What you built today

| # | Agent | Tools | Verified by |
|---|---|---|---|
| 1 | Code Auditor | radon · pylint · PyExamine | JSON contract |
| 2 | ML Auditor | MLScent | JSON contract |
| 3 | Refactorer | — | the QA gates |
| 4 | QA Verifier | ast · pytest · radon | *it is the verifier* |
| 5 | TD Classifier | — | gold labels |
| 6 | Triage Agent | — | human judgement |

## Scaling this up in the real world

| Today's toy | Production equivalent |
|---|---|
| `WorkflowState` dataclass | LangGraph state graphs · AutoGen conversations · CrewAI crews |
| a prompted general model | fine-tuned small models — cheaper *and* more consistent for narrow tasks |
| zero-shot TD classifier | **BEACon-TD / TD-Suite** fine-tuned transformers (13 debt types) |
| 4 hand-wrapped tools | **PyExamine** (49 metrics) · **MLScent** (76 ML anti-patterns) · full linter farms |
| 7 pytest tests as the gate | full CI: coverage thresholds, mutation testing, canary deploys |

## Papers & tools from today

- **PyExamine** — *MSR 2025* · `pip install code-quality-analyzer` · [github.com/KarthikShivasankar/python_smells_detector](https://github.com/KarthikShivasankar/python_smells_detector)
- **MLScent** — *CAIN 2025* · `pip install ml-code-smell-detector` · [arXiv:2502.18466](https://arxiv.org/abs/2502.18466)
- **BEACon-TD / TD-Suite** — *Journal of Systems and Software, 2025* · [github.com/KarthikShivasankar/text_classification](https://github.com/KarthikShivasankar/text_classification)
- *Enhancing Python Code Maintainability through LLM-Based Approaches* — Shivashankar & Martini, 2025
- Fowler, *Refactoring* (2nd ed.) — the smell taxonomy everything builds on
- Cunningham (1992) — the original "debt" metaphor. Two pages. Read it verbatim.

## The three sentences worth remembering

1. **Deterministic tools measure; the LLM interprets; a gate decides.**
2. **Never let the component that can hallucinate be the component that certifies it did not.**
3. **Capability grows through tools and verification, not through bigger models.**

In [ ]:
# Final bill for the whole workshop
cost_report()
print("\n📁 artefacts in", os.getcwd(), ":")
for p in sorted(pathlib.Path(".").glob("*")):
    print("  ", p.name)
print("\n🎉 Thanks for building with us! — Karthik & Adela · LLMA4SE 2026")